<a href="https://colab.research.google.com/github/AkramMoustafa/air_pollution_prediction/blob/main/AirPredictionModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install requests pandas

import os
import time
import math
import csv
import random
import requests
import pandas as pd
import json
import folium
from datetime import datetime, timedelta, timezone
from google.colab import userdata

API_KEY = userdata.get("OpenAQKey")

LAT, LON = 34.1808, -118.3090
RADIUS_M = 10_000

CORE_PARAMS = {"pm25", "pm10"}

INCLUDE_ALL_PARAMS = False

DAYS = 365 * 3

# Chunking:
CHUNK_DAYS = 90   #3 Month

# Sensor Paging
LOCATION_LIMIT = 10
MEAS_LIMIT = 1000
MAX_PAGES_LOC = 50
MAX_PAGES_MEAS = 50

# Rate Throttle
BASE_SLEEP = 1.00
JITTER = 0.2

BASE_URL = "https://api.openaq.org/v3"
HEADERS = {"X-API-Key": API_KEY}

# Output structure
RUN_TAG = f"openaq_{LAT}_{LON}_r{RADIUS_M}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUT_DIR = RUN_TAG
BIG_CSV_PATH = os.path.join(OUT_DIR, "all_measurements_long.csv")
SENSORS_CSV_PATH = os.path.join(OUT_DIR, "sensors_in_radius.csv")
FAIL_CSV_PATH = os.path.join(OUT_DIR, "failures.csv")

PART_DIR = os.path.join(OUT_DIR, "partitioned")
TRAIN_DIR = os.path.join(OUT_DIR, "train_partitioned")
TEST_DIR  = os.path.join(OUT_DIR, "test_partitioned")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PART_DIR, exist_ok=True)
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

# HTTP help
session = requests.Session()

def sleep_time(mult=1.0):
    time.sleep((BASE_SLEEP + random.random() * JITTER) * mult)

def get_json(endpoint, params=None, retries=6):
    url = f"{BASE_URL}{endpoint}"
    params = params or {}
    backoff = 1.0
    for attempt in range(retries):
        try:
            r = session.get(url, headers=HEADERS, params=params, timeout=60)
            print("GET", r.url, "->", r.status_code)
            # rate limit / errors
            if r.status_code in (429, 500, 502, 503, 504):
                sleep_time(mult=backoff)
                backoff *= 2
                continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print("Request error:", repr(e))
            if attempt == retries - 1:
                raise
            sleep_time(mult=backoff)
            backoff *= 2
    raise RuntimeError("Unreachable")

def paginate(endpoint, params, limit=1000, max_pages=50):
    out = []
    for page in range(1, max_pages + 1):
        p = dict(params)
        p["limit"] = limit
        p["page"] = page
        data = get_json(endpoint, p)
        results = data.get("results", [])
        if not results:
            break
        out.extend(results)
        if len(results) < limit:
            break
        sleep_time()
    return out

# Time window
dt_to = datetime.now(timezone.utc)
dt_from = dt_to - timedelta(days=DAYS)

def iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")

def iteration_chunks(start_dt, end_dt, chunk_days=30):
    cur = start_dt
    while cur < end_dt:
        next = min(cur + timedelta(days=chunk_days), end_dt)
        yield cur, next
        cur = next

# Output writers
BIG_HEADER = [
    "sensor_id","parameter","value",
    "datetime_utc","datetime_local",
    "location_id","location_name","location_lat","location_lon",
    "sensor_name","units"
]

# write csv header
with open(BIG_CSV_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(BIG_HEADER)

def partition_path(base_dir, parameter, sensor_id, dt_utc):
    # Partition by parameter/year/month for modeling
    if pd.isna(dt_utc):
        y, m = "unknown", "unknown"
    else:
        y = f"{dt_utc.year:04d}"
        m = f"{dt_utc.month:02d}"
    d = os.path.join(base_dir, f"parameter={parameter}", f"year={y}", f"month={m}")
    os.makedirs(d, exist_ok=True)
    return os.path.join(d, f"sensor_{sensor_id}.csv")

def append_rows_to_csv(path, header, rows):
    exists = os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if not exists:
            w.writerow(header)
        w.writerows(rows)

# 1) Find locations
locations = paginate(
    "/locations",
    params={"coordinates": f"{LAT},{LON}", "radius": RADIUS_M},
    limit=LOCATION_LIMIT,
    max_pages=MAX_PAGES_LOC
)
print(f"Locations found: {len(locations)}")

# 2) Extract sensors
sensor_rows = []
param_set = set()

for loc in locations:
    loc_coords = loc.get("coordinates") or {}
    for s in (loc.get("sensors") or []):
        param = s.get("parameter") or {}
        p_name = param.get("name") if isinstance(param, dict) else None
        if p_name:
            param_set.add(p_name)

        sensor_rows.append({
            "sensor_id": s.get("id"),
            "sensor_name": s.get("name"),
            "parameter": p_name,
            "units": (param.get("units") if isinstance(param, dict) else None),
            "location_id": loc.get("id"),
            "location_name": loc.get("name"),
            "location_lat": loc_coords.get("latitude"),
            "location_lon": loc_coords.get("longitude"),
        })

df_sensors = pd.DataFrame(sensor_rows).drop_duplicates("sensor_id").reset_index(drop=True)
df_sensors.to_csv(SENSORS_CSV_PATH, index=False)
print(f"Sensors found: {len(df_sensors)}")
print(f"Parameters in area (sample): {sorted(list(param_set))[:20]}")

if df_sensors.empty:
    raise ValueError("No sensors found in radius. Try different LAT/LON/RADIUS.")

if INCLUDE_ALL_PARAMS:
    params_to_pull = sorted([p for p in param_set if p])
else:
    params_to_pull = sorted([p for p in param_set if p in CORE_PARAMS])

if not params_to_pull:
    print("No params found (pm25/pm10). Falling back to default parameters.")
    params_to_pull = sorted([p for p in param_set if p])

print("Will pull parameters:", params_to_pull)

# Build sensor list that matches parameters
df_sensors_use = df_sensors[df_sensors["parameter"].isin(params_to_pull)].copy()
sensor_ids = df_sensors_use["sensor_id"].dropna().astype(int).tolist()
print(f"Using sensors in params_to_pull: {len(sensor_ids)}")

if not sensor_ids:
    raise ValueError("No matching sensors for parameters. Try INCLUDE_ALL_PARAMS=True or new area.")

# Map sensor_id
meta = df_sensors_use.set_index("sensor_id").to_dict(orient="index")

# 3) Pull measurements in chunks, stream to disk
failures = []

sensor_time_min = {}
sensor_time_max = {}

def update_minmax(sid, dt_utc):
    if pd.isna(dt_utc):
        return
    mn = sensor_time_min.get(sid)
    mx = sensor_time_max.get(sid)
    if mn is None or dt_utc < mn:
        sensor_time_min[sid] = dt_utc
    if mx is None or dt_utc > mx:
        sensor_time_max[sid] = dt_utc

for i, sid in enumerate(sensor_ids, 1):
    print(f"[{i}/{len(sensor_ids)}] sensor {sid}")

    s_meta = meta.get(sid, {})
    s_param = s_meta.get("parameter") or "unknown"

    # Pull in chunks
    for cstart, cend in iteration_chunks(dt_from, dt_to, CHUNK_DAYS):
        try:
            results = paginate(
                f"/sensors/{sid}/measurements",
                params={"datetime_from": iso(cstart), "datetime_to": iso(cend)},
                limit=MEAS_LIMIT,
                max_pages=MAX_PAGES_MEAS
            )

            # Convert  rows
            big_rows = []
            part_rows = []
            part_header = BIG_HEADER

            for r in results:
                dt = r.get("datetime") or {}
                dt_utc = dt.get("utc")
                dt_local = dt.get("local")

                row = [
                    sid,
                    r.get("parameter") or s_param,
                    r.get("value"),
                    dt_utc,
                    dt_local,
                    s_meta.get("location_id"),
                    s_meta.get("location_name"),
                    s_meta.get("location_lat"),
                    s_meta.get("location_lon"),
                    s_meta.get("sensor_name"),
                    s_meta.get("units"),
                ]
                big_rows.append(row)
                part_rows.append(row)

                parsed = pd.to_datetime(dt_utc, errors="coerce", utc=True)
                if not pd.isna(parsed):
                    update_minmax(sid, parsed)

            if big_rows:
                # append to main file
                append_rows_to_csv(BIG_CSV_PATH, BIG_HEADER, big_rows)

                # append to partition file
                first_dt = pd.to_datetime(big_rows[0][3], errors="coerce", utc=True)
                ppath = partition_path(PART_DIR, s_param, sid, first_dt)
                append_rows_to_csv(ppath, part_header, part_rows)

        except Exception as e:
            failures.append((sid, s_param, iso(cstart), iso(cend), str(e)))
            print("  FAILED chunk:", iso(cstart), "to", iso(cend), "->", e)

        sleep_time()

    sleep_time()

# Save failures
if failures:
    df_fail = pd.DataFrame(failures, columns=["sensor_id","parameter","chunk_from","chunk_to","error"])
    df_fail.to_csv(FAIL_CSV_PATH, index=False)
    print("Saved failures:", FAIL_CSV_PATH)

print("Done pulling. Big file:", BIG_CSV_PATH)

# 4) Build 80/20 time split

print("Creating train/test split (80/20) ...")

# cutoffs
cutoff = {}
for sid in sensor_time_min:
    mn = sensor_time_min[sid]
    mx = sensor_time_max.get(sid)
    if mx is None or mn is None or mx <= mn:
        continue
    span = (mx - mn)
    cutoff[sid] = mn + span * 0.8

# Stream main CSV and dispatch rows
train_count = 0
test_count = 0

TRAIN_BIG = os.path.join(OUT_DIR, "train_long.csv")
TEST_BIG  = os.path.join(OUT_DIR, "test_long.csv")

with open(TRAIN_BIG, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerow(BIG_HEADER)
with open(TEST_BIG, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerow(BIG_HEADER)

# Read file in chunks
for chunk in pd.read_csv(BIG_CSV_PATH, chunksize=200_000):
    chunk["datetime_utc_parsed"] = pd.to_datetime(chunk["datetime_utc"], errors="coerce", utc=True)
    # select rows
    train_rows = []
    test_rows = []
    for row in chunk.itertuples(index=False):
        sid = int(row.sensor_id)
        dtp = row.datetime_utc_parsed
        if pd.isna(dtp) or sid not in cutoff:
            # if no time info, shove into train (conservative)
            train_rows.append([row.sensor_id,row.parameter,row.value,row.datetime_utc,row.datetime_local,
                               row.location_id,row.location_name,row.location_lat,row.location_lon,row.sensor_name,row.units])
            continue
        if dtp <= cutoff[sid]:
            train_rows.append([row.sensor_id,row.parameter,row.value,row.datetime_utc,row.datetime_local,
                               row.location_id,row.location_name,row.location_lat,row.location_lon,row.sensor_name,row.units])
        else:
            test_rows.append([row.sensor_id,row.parameter,row.value,row.datetime_utc,row.datetime_local,
                              row.location_id,row.location_name,row.location_lat,row.location_lon,row.sensor_name,row.units])

    # append train/test big
    if train_rows:
        append_rows_to_csv(TRAIN_BIG, BIG_HEADER, train_rows)
        train_count += len(train_rows)
    if test_rows:
        append_rows_to_csv(TEST_BIG, BIG_HEADER, test_rows)
        test_count += len(test_rows)

    # partition
    def write_partition(rows, base_dir):
        # group by (parameter, sensor_id, year, month)
        df = pd.DataFrame(rows, columns=BIG_HEADER)
        df["dtp"] = pd.to_datetime(df["datetime_utc"], errors="coerce", utc=True)
        df["year"] = df["dtp"].dt.year.fillna(-1).astype(int)
        df["month"] = df["dtp"].dt.month.fillna(-1).astype(int)

        for (param, sid, y, m), g in df.groupby(["parameter","sensor_id","year","month"]):
            ystr = "unknown" if y == -1 else f"{y:04d}"
            mstr = "unknown" if m == -1 else f"{m:02d}"
            d = os.path.join(base_dir, f"parameter={param}", f"year={ystr}", f"month={mstr}")
            os.makedirs(d, exist_ok=True)
            p = os.path.join(d, f"sensor_{sid}.csv")
            append_rows_to_csv(p, BIG_HEADER, g[BIG_HEADER].values.tolist())

    if train_rows:
        write_partition(train_rows, TRAIN_DIR)
    if test_rows:
        write_partition(test_rows, TEST_DIR)

print(f"Train rows: {train_count:,} | Test rows: {test_count:,}")
print("Train big:", TRAIN_BIG)
print("Test big :", TEST_BIG)
print("Partitioned full:", PART_DIR)
print("Partitioned train:", TRAIN_DIR)
print("Partitioned test :", TEST_DIR)

summary = {
    "run_time_utc": datetime.now(timezone.utc).isoformat(),
    "area": {"lat": LAT, "lon": LON, "radius_m": RADIUS_M},
    "days_back": DAYS,
    "chunk_days": CHUNK_DAYS,
    "params_to_pull": params_to_pull if "params_to_pull" in globals() else None,
    "counts": {
        "locations": len(locations) if "locations" in globals() else None,
        "sensors_total": int(df_sensors.shape[0]) if "df_sensors" in globals() else None,
        "sensors_used": len(sensor_ids) if "sensor_ids" in globals() else None,
        "failures": len(failures) if "failures" in globals() else None,
    },
    "files": {
        "big_csv": BIG_CSV_PATH,
        "sensors_csv": SENSORS_CSV_PATH,
        "failures_csv": FAIL_CSV_PATH if os.path.exists(FAIL_CSV_PATH) else None,
        "train_big": TRAIN_BIG,
        "test_big": TEST_BIG,
        #"pm25_hourly": os.path.join(OUT_DIR, "hourly_pm25_long.csv"),
        #"pm10_hourly": os.path.join(OUT_DIR, "hourly_pm10_long.csv"),
        #"pm25_health": os.path.join(OUT_DIR, "pm25_sensor_health.csv"),
        #"pm25_area_aggregates": os.path.join(OUT_DIR, "pm25_area_aggregates_hourly.csv"),
    }
}

sum_path = os.path.join(OUT_DIR, "run_summary.json")
with open(sum_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)

print("Saved:", sum_path)

map = folium.Map(location=[LAT, LON], zoom_start=12, control_scale=True)

folium.Circle(
    location=[LAT, LON],
    radius=RADIUS_M,
    color="blue",
    fill=False,
    weight=2,
    tooltip=f"Query center (r={RADIUS_M}m)"
).add_to(map)

param_colors = {"pm25": "red", "pm2.5": "red", "pm10": "orange", "pm1": "purple"}

df_map = df_sensors_use.dropna(subset=["location_lat", "location_lon"]).copy()

for _, r in df_map.iterrows():
    p = str(r.get("parameter", "unknown"))
    color = param_colors.get(p, "gray")
    tooltip = f"id={r.sensor_id} | {p} | {r.location_name} | units={r.units}"
    folium.CircleMarker(
        location=[float(r.location_lat), float(r.location_lon)],
        radius=5,
        color=color,
        fill=True,
        fill_opacity=0.8,
        tooltip=tooltip
    ).add_to(map)

map.save(SENSOR_MAP)
print("Saved map:", SENSOR_MAP)

map



GET https://api.openaq.org/v3/locations?coordinates=34.1808%2C-118.309&radius=10000&limit=10&page=1 -> 200
GET https://api.openaq.org/v3/locations?coordinates=34.1808%2C-118.309&radius=10000&limit=10&page=2 -> 200
GET https://api.openaq.org/v3/locations?coordinates=34.1808%2C-118.309&radius=10000&limit=10&page=3 -> 200
GET https://api.openaq.org/v3/locations?coordinates=34.1808%2C-118.309&radius=10000&limit=10&page=4 -> 200
Locations found: 32
Sensors found: 165
Parameters in area (sample): ['no', 'no2', 'nox', 'o3', 'pm1', 'pm10', 'pm25', 'relativehumidity', 'temperature', 'um003']
Will pull parameters: ['pm10', 'pm25']
Using sensors in params_to_pull: 57
[1/57] sensor 24000
GET https://api.openaq.org/v3/sensors/24000/measurements?datetime_from=2023-03-17T17%3A17%3A37Z&datetime_to=2023-06-15T17%3A17%3A37Z&limit=1000&page=1 -> 200
GET https://api.openaq.org/v3/sensors/24000/measurements?datetime_from=2023-03-17T17%3A17%3A37Z&datetime_to=2023-06-15T17%3A17%3A37Z&limit=1000&page=2 -> 200

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
